In [86]:
import pandas as pd

df_ground_truth = pd.read_csv('data/ground_truth.csv')

ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[0]





{'question': 'I just found this course late — can I still sign up and follow along, or is it too late?',
 'document': '74eb249bbf'}

In [87]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents_llm = []

for doc in documents:
  if doc['course'] == 'llm-zoomcamp':
    documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 118}]


In [88]:
boost = {'question': 3.0}

index.search("Can I still take the course?", num_results=5, boost_dict=boost)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone proj

In [89]:
def text_search(query):
  boost_dict = {'question': 3.0, "section": 0.5}

  return index.search(query, num_results=5, boost_dict=boost_dict)

In [90]:
q = documents[0]
results = text_search(q['question'])

In [91]:
doc_id = q['id']
doc_id

'74eb249bbf'

In [92]:
for r in results:
  print(f'{r["id"]} == {doc_id}: {r["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
04919992b3 == 74eb249bbf: False


In [93]:
# relevance matrix
#    1 2 3 4 5
# q1 1 0 0 0 0
# q2 0 1 0 0 0
# q3 0 0 1 0 0
# q4 0 0 0 1 0
# q5 0 0 0 0 1

relevance = []

for d in results:
  relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [94]:
def compute_relevance_text(q):
  doc_id = q["document"]
  results = text_search(query=q["question"])

  relevance = [int(r["id"] == doc_id) for r in results]
  return relevance

In [95]:
q = documents[99]
compute_relevance_text(q)

KeyError: 'document'

In [ ]:
from tqdm.auto import tqdm

def compute_relavance_total_text(ground_truth):
  relevance_total = []
  for q in tqdm(ground_truth):
    relevance_total.append(compute_relevance_text(q))
  return relevance_total




In [ ]:
all_relevance = compute_relavance_total_text(ground_truth)
# ground_truth[0]


  0%|          | 0/590 [00:00<?, ?it/s]

In [ ]:
all_relevance[:30]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [ ]:
def compute_relevance(q, search_fn):
  doc_id = q["document"]
  results = search_fn(query=q["question"])

  return [int(r["id"] == doc_id) for r in results]

def compute_relavance_total(ground_truth, search_fn):
  relevance_total = []
  for q in tqdm(ground_truth):
    relevance_total.append(compute_relevance(q, search_fn))
  return relevance_total


In [ ]:
all_relevance = compute_relavance_total(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

In [ ]:
all_relevance[0]

[1, 0, 0, 0, 0]

In [ ]:
# hit rate if there is at least one relevant document per query
def hit_rate(relevance_total):
  return sum(1 for r in relevance_total if any(r)) / len(relevance_total)

hit_rate(all_relevance)




0.788135593220339

In [ ]:
# mrr
def reciprocal_rank(line):
  for rank, relevance in enumerate(line):
    if relevance:
      return 1 / (rank + 1)
  return 0

def mrr(relevance_total):
  return sum(reciprocal_rank(line) for line in relevance_total) / len(relevance_total)

print(mrr(all_relevance))


0.6692090395480226


In [96]:
def evaluate(ground_truth, search_fn):
  relevance_total = compute_relavance_total(ground_truth, search_fn)
  return {
    "hit_rate": hit_rate(relevance_total),
    "mrr": mrr(relevance_total)
  }

evaluate(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8084745762711865, 'mrr': 0.6846327683615819}

In [100]:
# lets try to improve the search function
def text_search_v2(query):
  boost_dict = {'question': 2.0, "section": 0.5}

  return index.search(query, num_results=5, boost_dict=boost_dict)

def text_search_v3(query):
  boost_dict = {'question': 1.0, "section": 0.5}

  return index.search(query, num_results=5, boost_dict=boost_dict)

In [99]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8322033898305085, 'mrr': 0.7106497175141243}

In [101]:
evaluate(ground_truth, text_search_v3)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8559322033898306, 'mrr': 0.7522598870056497}

In [105]:
def search_boost(query, question_boost):
  boost_dict = {'question': question_boost, "section": 0.5}

  return index.search(query, num_results=5, boost_dict=boost_dict)

evaluate(ground_truth, lambda query: search_boost(query, 1.0))



  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8559322033898306, 'mrr': 0.7522598870056497}

In [108]:
evaluate(ground_truth, lambda query: search_boost(query, 0.9))

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8593220338983051, 'mrr': 0.7592090395480227}

In [111]:
def search_boost(query, question_boost, answer_boost, section_boost):
  boost_dict = {
    'question': question_boost,
    'answer': answer_boost,
    'section': section_boost
  }
  filter_dict = {
    'course': 'llm-zoomcamp'
  }
  return index.search(query, num_results=5, boost_dict=boost_dict, filter_dict=filter_dict)

results = []

for question_boost in [0.5, 1.0, 2.0, 5.0]:
  for answer_boost in [0.5, 1.0, 2.0, 4.0, 10.0]:
    for section_boost in [0.1, 0.2, 0.5]:
      result = evaluate(ground_truth, lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boost(query, question_boost, answer_boost, section_boost))
      
      results.append({
        'question_boost': question_boost,
        'answer_boost': answer_boost,
        'section_boost': section_boost,
        'hit_rate': result['hit_rate'],
        'mrr': result['mrr']
      })

results




  0%|          | 0/590 [00:00<?, ?it/s]

  0%|          | 0/590 [00:00<?, ?it/s]

  0%|          | 0/590 [00:00<?, ?it/s]

  0%|          | 0/590 [00:00<?, ?it/s]

  0%|          | 0/590 [00:00<?, ?it/s]

...

  0%|          | 0/590 [00:00<?, ?it/s]

  0%|          | 0/590 [00:00<?, ?it/s]

  0%|          | 0/590 [00:00<?, ?it/s]

  0%|          | 0/590 [00:00<?, ?it/s]

[{'question_boost': 0.5,
  'answer_boost': 0.5,
  'section_boost': 0.1,
  'hit_rate': 0.9050847457627119,
  'mrr': 0.7881073446327684},
 {'question_boost': 0.5,
  'answer_boost': 0.5,
  'section_boost': 0.2,
  'hit_rate': 0.8728813559322034,
  'mrr': 0.7614124293785312},
 {'question_boost': 0.5,
  'answer_boost': 0.5,
  'section_boost': 0.5,
  'hit_rate': 0.8067796610169492,
  'mrr': 0.6988418079096045},
 {'question_boost': 0.5,
  'answer_boost': 1.0,
  'section_boost': 0.1,
  'hit_rate': 0.9322033898305084,
  'mrr': 0.8301977401129943},
 {'question_boost': 0.5,
  'answer_boost': 1.0,
  'section_boost': 0.2,
  'hit_rate': 0.9135593220338983,
  'mrr': 0.813728813559322},
 {'question_boost': 0.5,
  'answer_boost': 1.0,
  'section_boost': 0.5,
  'hit_rate': 0.847457627118644,
  'mrr': 0.7577118644067797},
 {'question_boost': 0.5,
  'answer_boost': 2.0,
  'section_boost': 0.1,
  'hit_rate': 0.9542372881355933,
  'mrr': 0.834774011299435},
 {'question_boost': 0.5,
  'answer_boost': 2.0,
  '

In [112]:
df_results = pd.DataFrame(results)
df_results.sort_values(by='mrr', ascending=False).head(10)

,question_boost,answer_boost,section_boost,hit_rate,mrr
24,1.0,4.0,0.1,0.957627,0.835113
6,0.5,2.0,0.1,0.954237,0.834774
25,1.0,4.0,0.2,0.954237,0.834774
43,2.0,10.0,0.2,0.954237,0.832910
21,1.0,2.0,0.1,0.937288,0.832458
40,2.0,4.0,0.2,0.937288,0.832458
59,5.0,10.0,0.5,0.937288,0.832458
42,2.0,10.0,0.1,0.957627,0.832175
57,5.0,10.0,0.1,0.942373,0.832090
58,5.0,10.0,0.2,0.942373,0.831780
